In [51]:
import warnings
from fractions import Fraction
 
import math
import time
import numpy as np
from pyscipopt import Model, quicksum
from tqdm import trange

_NEG = np.iinfo(np.int64).min // 4  

In [52]:
def rationalize(values, max_denominator, name, snap_tol=1e-9):
    out, worst = [], 0.0
    for v in np.atleast_1d(np.asarray(values, dtype=float)).ravel():
        if not math.isfinite(v):
            raise ValueError(f"{name} contains non-finite value {v}")
        f = Fraction(float(v)).limit_denominator(max_denominator)
        worst = max(worst, abs(float(f) - float(v)))
        out.append(f)
    if worst > snap_tol:
        warnings.warn(
            f"{name}: rational snapping moved a value by {worst:.2e} "
            f"(> {snap_tol:.0e}). Raise max_denominator, or pass values that "
            f"are exactly representable as simple fractions.", stacklevel=3)
    return out

In [53]:
def block_response(Uint, valid, members):
    total = Uint[list(members)].sum(axis=0)
    return np.where(valid, total, _NEG).argmax(axis=1)

def block_loss(Uint, valid, Lmat, Dint, Pint, members):
    best = block_response(Uint, valid, members)
    rows = np.arange(Uint.shape[1])
    total = 0
    for i in members:
        hit = np.nonzero(Lmat[i, rows, best])[0]
        if hit.size:
            total += Pint[i] * sum(Dint[r] for r in hit)
    return total

def partition_loss(Uint, valid, Lmat, Dint, Pint, partition):
    return sum(block_loss(Uint, valid, Lmat, Dint, Pint, list(b))
               for b in partition)

In [54]:
def build_action_sets(t, X):
    n, m = len(t), len(X)
    acts = np.empty((m, n + 1), dtype=object)
    nact = np.zeros(m, dtype=np.int64)
    for xi, x in enumerate(X):
        A = sorted({x} | {ti for ti in t if ti > x})
        nact[xi] = len(A)
        for ai, a in enumerate(A):
            acts[xi, ai] = a
    return acts, nact

In [55]:
def lcm_denominators(fracs):
    q = 1
    for f in fracs:
        q = q * f.denominator // math.gcd(q, f.denominator)
    return q

def build_integer_tables(t, pi, tau, c, X, acts, nact):
    n, m = len(t), len(X)
    Q = lcm_denominators(pi) * c.denominator * lcm_denominators(list(t) + list(X))
    limit = np.iinfo(np.int64).max // max(n, 1)
 
    Uint = np.zeros((n, m, n + 1), dtype=np.int64)
    Lmat = np.zeros((n, m, n + 1), dtype=np.int8)
    valid = np.zeros((m, n + 1), dtype=bool)
    for xi in range(m):
        label = 1 if X[xi] >= tau else 0
        for ai in range(int(nact[xi])):
            a = acts[xi, ai]
            valid[xi, ai] = True
            cost = c * max(a - X[xi], Fraction(0))
            for i in range(n):
                h = 1 if a >= t[i] else 0
                u = pi[i] * (h - cost) * Q
                if u.denominator != 1:
                    raise AssertionError("integer scaling failed")
                if abs(int(u)) > limit:
                    raise OverflowError(
                        "scaled utilities exceed int64 range; lower "
                        "max_denominator or use simpler rational inputs")
                Uint[i, xi, ai] = int(u)
                Lmat[i, xi, ai] = 1 if h != label else 0
    return Uint, Lmat, valid, Q

In [56]:
def certified_epsilon(Q, c, X, acts, nact):
    Cmax = Fraction(0)
    for xi in range(len(X)):
        for ai in range(int(nact[xi])):
            Cmax = max(Cmax, c * max(acts[xi, ai] - X[xi], Fraction(0)))
    return Fraction(1, 2) if Cmax == 0 else Fraction(1, 2) / (Q * Cmax)

In [57]:
def perturbed_utilities(t, pi, c, X, acts, nact, eps):
    n, m = len(t), len(X)
    Uf = np.zeros((n, m, n + 1))
    for xi in range(m):
        for ai in range(int(nact[xi])):
            cost = c * max(acts[xi, ai] - X[xi], Fraction(0))
            for i in range(n):
                h = 1 if acts[xi, ai] >= t[i] else 0
                Uf[i, xi, ai] = float(pi[i] * (h - (1 + eps) * cost))
    return Uf

In [58]:
def loss_coefficients(pi, Lmat):
    n, m, amax = Lmat.shape
    piL = np.zeros((n, m, amax))
    for i in range(n):
        piL[i] = float(pi[i]) * Lmat[i]
    return piL

In [59]:
def tight_epsilon(Uint, valid, nact, Q, Cmax, max_subsets=1 << 15):
    n = Uint.shape[0]
    if (1 << n) > max_subsets:
        return None
    best = None
    for mask in range(1, 1 << n):
        members = [i for i in range(n) if mask >> i & 1]
        tot = Uint[members].sum(axis=0)
        d = np.abs(tot[:, :, None] - tot[:, None, :])
        ok = valid[:, :, None] & valid[:, None, :]
        d = np.where(ok & (d > 0), d, np.iinfo(np.int64).max)
        v = int(d.min())
        if v != np.iinfo(np.int64).max and (best is None or v < best):
            best = v
    if best is None:
        return Fraction(1, 2)
    return Fraction(best, Q) / (2 * Cmax) if Cmax > 0 else Fraction(1, 2)

In [60]:
def max_cost(c, X, acts, nact):
    Cmax = Fraction(0)
    for xi in range(len(X)):
        for ai in range(int(nact[xi])):
            Cmax = max(Cmax, c * max(acts[xi, ai] - X[xi], Fraction(0)))
    return Cmax

In [61]:
def big_m_table(Uf, nact):
    n, m, amax = Uf.shape
    M = np.zeros((m, amax, amax))
    for xi in range(m):
        k = int(nact[xi])
        for ai in range(k):
            for api in range(k):
                if ai != api:
                    M[xi, ai, api] = np.maximum(
                        Uf[:, xi, api] - Uf[:, xi, ai], 0.0).sum()
    return M

In [62]:
def solve_miqp(n, m, Uf, nact, piL, Dfloat, time_limit=None, verbose=False,
               gap=0.0, threads=None):
    M = big_m_table(Uf, nact)
    model = Model("partition_miqp")
    if not verbose:
        model.hideOutput()
    if time_limit:
        model.setParam("limits/time", float(time_limit))
    if threads:
        model.setParam("parallel/maxnthreads", int(threads))
    model.setParam("limits/gap", float(gap))
    model.setParam("numerics/feastol", 1e-9)
    model.setParam("numerics/epsilon", 1e-10)
 
    x = {(i, j): model.addVar(vtype="B", name=f"x_{i}_{j}")
         for i in range(n) for j in range(n)}
    y = {(xi, ai, j): model.addVar(vtype="B", name=f"y_{xi}_{ai}_{j}")
         for xi in range(m) for ai in range(int(nact[xi])) for j in range(n)}
 
    for i in range(n):
        model.addCons(quicksum(x[i, j] for j in range(n)) == 1)
 
    for i in range(n):
        for j in range(i + 1, n):
            model.addCons(x[i, j] == 0)
    for i in range(1, n):
        for j in range(1, i + 1):
            model.addCons(x[i, j] <= quicksum(x[ip, j - 1] for ip in range(i)))
 
    for xi in range(m):
        for j in range(n):
            model.addCons(
                quicksum(y[xi, ai, j] for ai in range(int(nact[xi]))) == 1)
 
    for xi in range(m):
        k = int(nact[xi])
        for j in range(n):
            for ai in range(k):
                ua = quicksum(float(Uf[i, xi, ai]) * x[i, j] for i in range(n))
                for api in range(k):
                    if api == ai:
                        continue
                    uap = quicksum(float(Uf[i, xi, api]) * x[i, j]
                                   for i in range(n))
                    model.addCons(
                        ua >= uap - float(M[xi, ai, api]) * (1 - y[xi, ai, j]))
 
    terms = []
    for xi in range(m):
        d = float(Dfloat[xi])
        if d == 0.0:
            continue
        for j in range(n):
            for ai in range(int(nact[xi])):
                for i in range(n):
                    coef = d * float(piL[i, xi, ai])
                    if coef != 0.0:
                        terms.append(coef * x[i, j] * y[xi, ai, j])
 
    q = model.addVar(lb=0.0, ub=1.0, name="q")
    if terms:
        model.addCons(q >= quicksum(terms))
    model.setObjective(q, "minimize")
 
    model.optimize()
    status = model.getStatus()
    if status not in ("optimal", "gaplimit", "timelimit") or \
            model.getNSols() == 0:
        raise RuntimeError(f"SCIP did not solve (status: {status})")
 
    sol = model.getBestSol()
    blocks = {}
    for i in range(n):
        for j in range(n):
            if sol[x[i, j]] > 0.5:
                blocks.setdefault(j, []).append(i)
    part = sorted(sorted(b) for b in blocks.values())
    return part, model.getNVars(), model.getNConss()

In [63]:
def find_optimal_partition_qp(thresholds, priors, true_threshold, c, X, weights=None, epsilon="auto", max_denominator=10 ** 6, time_limit=None, verbose=False, gap=0.0, threads=None):
    t_raw = np.asarray(thresholds, dtype=float).ravel()
    p_raw = np.asarray(priors, dtype=float).ravel()
    X_in = np.asarray(X, dtype=float).ravel()
    X_raw = np.unique(X_in)
    n, m = len(t_raw), len(X_raw)
    if len(p_raw) != n:
        raise ValueError(f"priors has length {len(p_raw)}, thresholds has {n}")
    if np.any(p_raw <= 0):
        raise ValueError("priors must be strictly positive")
    if n == 0 or m == 0:
        raise ValueError("thresholds and X must be non-empty")
 
    t = rationalize(t_raw, max_denominator, "thresholds")
    pi = rationalize(p_raw, max_denominator, "priors")
    sp = sum(pi)
    pi = [p / sp for p in pi]
    tau = rationalize([true_threshold], max_denominator, "true_threshold")[0]
    cc = rationalize([c], max_denominator, "c")[0]
    Xf = rationalize(X_raw, max_denominator, "X")
 
    if weights is None:
        D = [Fraction(1, m)] * m
    else:
        w_raw = np.asarray(weights, dtype=float).ravel()
        if len(w_raw) != len(X_in):
            raise ValueError("weights must have the same length as X")
        folded = {}
        for v, wv in zip(X_in, w_raw):
            folded[v] = folded.get(v, 0.0) + wv
        w_aligned = np.array([folded[v] for v in X_raw])
        if np.any(w_aligned < 0):
            raise ValueError("weights must be non-negative")
        D = rationalize(w_aligned, max_denominator, "weights")
        sd = sum(D)
        D = [d / sd for d in D]
 
    acts, nact = build_action_sets(t, Xf)
    Uint, Lmat, valid, Q = build_integer_tables(t, pi, tau, cc, Xf, acts, nact)
 
    if isinstance(epsilon, str):
        Cmax = max_cost(cc, Xf, acts, nact)
        eps = None
        if epsilon in ("auto", "tight"):
            eps = tight_epsilon(Uint, valid, nact, Q, Cmax)
        if eps is None:
            eps = certified_epsilon(Q, cc, Xf, acts, nact)
        if float(eps) < 1e-7:
            warnings.warn(
                f"certified epsilon is {float(eps):.2e}, small enough that the "
                f"solver's integrality tolerance may swamp it; the reported "
                f"loss may be below the true optimum. Use simpler rational "
                f"inputs, or cross-check with the exact DP (check=True).",
                stacklevel=2)
    else:
        eps = Fraction(float(epsilon)).limit_denominator(10 ** 12)
 
    Uf = perturbed_utilities(t, pi, cc, Xf, acts, nact, eps)
    piL = loss_coefficients(pi, Lmat)
    Dfloat = np.array([float(d) for d in D])
 
    part, n_vars, n_constrs = solve_miqp(n, m, Uf, nact, piL, Dfloat,
                                         time_limit, verbose, gap, threads)
 
    Ld, Lp = lcm_denominators(D), lcm_denominators(pi)
    Dint = [int(d * Ld) for d in D]
    Pint = [int(p * Lp) for p in pi]
    loss_int = partition_loss(Uint, valid, Lmat, Dint, Pint, part)
 
    loss_fraction = Fraction(int(loss_int), int(Ld * Lp))
 
    err = np.zeros(n)
    mass = []
    rows = np.arange(m)
    for block in part:
        mass.append(float(sum(pi[i] for i in block)))
        best = block_response(Uint, valid, list(block))
        for i in block:
            hit = np.nonzero(Lmat[i, rows, best])[0]
            err[i] = float(sum(D[xi] for xi in hit))
 
    return {"loss": float(loss_fraction),
            "partition": part,
            "per_classifier_error": err,
            "block_mass": np.array(mass),
            "method": "miqp",
            "epsilon": float(eps),
            "n_vars": n_vars,
            "n_constrs": n_constrs,
            "priors_normalized": np.array([float(p) for p in pi]),
            "loss_fraction": loss_fraction}

In [ ]:
rng = np.random.default_rng(11)
ok = bad = 0
N = range(4, 10)
M = [51, 101, 151, 201]
tt = 0.1
c = 0.1

res_stats = {"n": [], "m": [], "counter": [], "thresholds": [], "priors": [], "c": [], "tt": [], "alg": [], "time": [], "partition": [], "loss": []}

for n in N:
    for m in M:
        X = np.linspace(0, 1, m)
        for counter in trange(100, desc=f"[n: {n}] [m: {m}]"):
            t = np.sort(rng.choice(np.arange(1, 20) / 20,size = n,replace = False))
            p = rng.integers(1, 9, size = n).astype(float)
            

            start_time = time.time()
            res = find_optimal_partition_qp(t, p, tt, c, X)
            end_time = time.time()

            res_stats["n"].append(n)
            res_stats["m"].append(m)
            res_stats["counter"].append(counter)
            res_stats["thresholds"].append(t)
            res_stats["priors"].append(res["priors_normalized"])
            res_stats["c"].append(c)
            res_stats["tt"].append(tt)
            res_stats["alg"].append("miqp")
            res_stats["time"].append(end_time - start_time)
            res_stats["partition"].append(res["partition"])
            res_stats["loss"].append(res["loss"])

In [67]:
import pandas as pd
import plotly.express as px

In [68]:
df = pd.DataFrame(res_stats)
df.to_csv("miqp.csv")

In [73]:
df_agg = df[df["n"] <6].groupby(["n", "m"], as_index=False).mean(1)
df_agg

,n,m,counter,c,tt,time,loss
0,4,51,49.5,0.1,0.1,0.176616,0.098039
1,4,101,49.5,0.1,0.1,0.438033,0.099010
2,4,151,49.5,0.1,0.1,0.793881,0.099338
3,4,201,49.5,0.1,0.1,1.267811,0.099502
4,5,51,49.5,0.1,0.1,0.499336,0.098039
5,5,101,49.5,0.1,0.1,1.302733,0.099010
6,5,151,49.5,0.1,0.1,2.332615,0.099338
7,5,201,49.5,0.1,0.1,3.896760,0.099502


In [76]:
px.line(df_agg, x="m", y="time", facet_col="n", labels={"time": "avg time (s)"}, markers=True)